# QwenAPI LLMOps demo

This sanitized notebook presents the reusable engineering pieces extracted from a private Kaggle experiment. It is intentionally offline-friendly: no API keys, tunnel tokens, or network calls are required.

## 1. Scope and portfolio framing

The project combines quantized Qwen serving, a FastAPI gateway, source-aware agentic search, and operational safeguards. It should be presented as a GPU-serving/LLMOps demonstration—not as a production SLA. The original Kaggle notebook remains private.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

print('Demo mode: local policy, cache, and mocked search only')

## 2. Architecture and request flow

A client authenticates to the gateway. The policy classifier decides whether live grounding is mandatory. The agent runner can call Roman Search, normalize and cache evidence, then return a cited answer through the model backend. See [`docs/architecture.mmd`](../docs/architecture.mmd) for the editable diagram.

In [ ]:
from qwenapi.config import Settings

settings = Settings()
settings.public_summary()

## 3. High-stakes grounding policy

Legal, medical, financial, cybersecurity, government, and public-safety questions are forced into live-evidence mode. Unknown policy values fail safe to required grounding.

In [ ]:
from qwenapi.policy import classify_query

examples = [
    ('What does Philippine law say about contracts?', 'off'),
    ('Summarize this supplied project brief', 'source_only'),
]
for query, mode in examples:
    print(classify_query(query, mode))

## 4. Search hygiene: canonicalization, dedupe, and TTL cache

Search results are sorted by score, capped per domain, and cached with category-specific TTLs to reduce cost and duplicate work.

In [ ]:
from qwenapi.search import SearchCache, SearchResult, canonical_url, dedupe_results

rows = [
    SearchResult('Primary source', 'HTTPS://example.com/a/#fragment', score=1.0),
    SearchResult('Duplicate domain', 'https://example.com/b', score=0.9),
    SearchResult('Other domain', 'https://docs.example.org/a', score=0.8),
]
print(canonical_url(rows[0].url))
print([item.title for item in dedupe_results(rows)])
cache = SearchCache(max_entries=2)
cache.put('demo', 'general', 'basic', rows)
print('cache hit:', cache.get('demo', 'general', 'basic') is not None)

## 5. Mocked agentic tool loop

The model backend is a protocol, so the orchestration can be tested without a GPU. This mock makes one `web_search` call, appends evidence, and then produces a final answer.

In [ ]:
import asyncio
from qwenapi.agent import AgentRunner

class DemoBackend:
    def __init__(self): self.calls = 0
    async def complete(self, messages, tools=None):
        self.calls += 1
        if tools and self.calls == 1:
            return {'message': {'tool_calls': [{'function': {'name': 'web_search', 'arguments': {'query': 'Qwen3.5 release'}}}]}}
        return {'message': {'content': 'Answer with the retrieved evidence [S1].'}}

class DemoSearch:
    async def search(self, query, *, category, depth='basic', limit=5):
        return [SearchResult('Demo source', 'https://example.com/qwen', 'A safe fixture.', 'S1', 1.0)]

demo = asyncio.run(AgentRunner(DemoBackend(), DemoSearch(), max_search_calls=3).run([{'role': 'user', 'content': 'Explain Qwen3.5'}]))
print(demo.content, demo.search_calls, len(demo.sources))

## 6. Observed private Kaggle trace

The original dual-T4 run used a Qwen3.5 4B quantized GGUF model, 32K context per slot, and four generation slots. One legal trace made 12 search calls, collected 46 sources, and completed in approximately 556 seconds. Roman Search latency varied, so this is an operational trace—not a general throughput claim.

In [ ]:
observed = {
    'hardware': '2 x NVIDIA Tesla T4',
    'context_per_slot': 32768,
    'generation_slots': 4,
    'agentic_search_calls': 12,
    'sources_collected': 46,
    'elapsed_seconds': 556,
}
observed

## 7. Security checklist

Before a public release: rotate any historical credentials, keep the Kaggle notebook private, remove revealing output, apply authentication to every API route, pin and verify downloaded binaries, use safe archive extraction, and avoid secrets in command-line arguments.

In [ ]:
checklist = [
    'No real credentials in this notebook',
    'Demo makes no network calls by default',
    'Historical keys must be rotated before public release',
    'Run tests and secret scanning in CI',
]
for item in checklist: print('[ ]', item)

## 8. Packaging and honest attribution

For a portfolio, pair this repository with a short README, the architecture diagram, tests, benchmark notes, and a link to the private Kaggle experiment only when reviewers have access. Describe the work as AI-assisted development: requirements were specified, coding agents were directed, changes were reviewed, tests were validated, and the service was operated.